In [1]:
import os
THREADS = "32"

# 1. Numba Threading
#os.environ["NUMBA_THREADING_LAYER"] = "tbb"
os.environ["NUMBA_NUM_THREADS"] = THREADS

# 2. NumPy backend Threading (copre MKL, OpenBLAS e OpenMP standard)
os.environ["OMP_NUM_THREADS"] = THREADS
os.environ["MKL_NUM_THREADS"] = THREADS
os.environ["OPENBLAS_NUM_THREADS"] = THREADS
os.environ["NUMEXPR_NUM_THREADS"] = THREADS

In [2]:

import sys

# 1. Definisci il percorso del worktree del main
repo_path = '/home/emaragliano/Work/Projects/Dottorato/baorecon_main'

# [Opzionale] Se i sorgenti sono dentro una cartella 'src', decommenta la riga sotto:
# repo_path = os.path.join(repo_path, 'src')

# Verifica di sicurezza: la cartella esiste davvero?
if not os.path.exists(repo_path):
    raise FileNotFoundError(f"Attenzione! Il percorso {repo_path} non esiste. Controlla trattini/underscore.")

# 2. Rimuovi COMPLETAMENTE le vecchie importazioni dalla cache di Python
# Questo impedisce a Python di usare il vecchio branch se la sessione è rimasta attiva
moduli_da_cancellare = [mod for mod in sys.modules if mod.startswith('zeldareco')]
for mod in moduli_da_cancellare:
    del sys.modules[mod]

# 3. Inserisci il percorso in cima a sys.path
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# 4. Esegui l'import e verifica la provenienza
import zeldareco
print(f"Successo! Modulo caricato da: {zeldareco.__file__}")

/farmdisk1/emaragliano/miniconda3/envs/BAOFit/lib/python3.10/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


Successo! Modulo caricato da: /home/emaragliano/Work/Projects/Dottorato/baorecon_main/zeldareco/__init__.py


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt

from zeldareco.pipeline import ReconstructionPipeline

In [4]:
pipeline = ReconstructionPipeline(
    config_file='bao_pipeline_example.yaml',
)

2026-06-19 21:00:14,156 - zeldareco.io.config - INFO - Loaded reconstruction config from bao_pipeline_example.yaml


In [5]:
saved_paths = pipeline.run()

2026-06-19 21:00:14,206 - zeldareco.pipeline.bao_pipeline - INFO - UPDATED VERSION OF THE PIPELINE! WILL SAVE OUTPUTS AT RUNTIME


2026-06-19 21:00:14,630 - zeldareco.io.catalog_io - INFO - Loaded catalogs: data=736756, random=37694049
2026-06-19 21:00:21,450 - zeldareco.utils.formatters - INFO - Rectangular box not yet supported. Setting box size to [2480.7522, 2480.7522, 2480.7522] based on maximum separation in positions with padding 100.0.
2026-06-19 21:00:25,826 - zeldareco.utils.formatters - INFO - Setting box centre to [ 871.27384843 1132.71534599 1658.36695932] from position range
2026-06-19 21:00:26,444 - zeldareco.BAOreconstruction.bao_reconstructor - INFO - mean data pos: [ 871.45266791 1137.85756274 1732.4540458 ]
2026-06-19 21:00:27,011 - zeldareco.BAOreconstruction.bao_reconstructor - INFO - mean random pos: [ 876.04878978 1140.20821273 1730.73427982]
2026-06-19 21:00:27,013 - zeldareco.BAOreconstruction.bao_reconstructor - INFO - Starting BAO reconstruction (MULTIGRID)...
2026-06-19 21:00:27,013 - zeldareco.BAOreconstruction.bao_reconstructor - INFO - Reconstruction type: rec-sym, RSD space: Redshif

In [6]:
saved_paths.keys()

dict_keys(['data_catalog', 'random_catalog', 'metadata'])

In [ ]:
kakakaka

# Read outputs

In [7]:
# read saved catalogs
from astropy.io import fits
rec_data = fits.open(saved_paths['data_catalog'])
grid_potential = fits.open(saved_paths['grid_potential'])



KeyError: 'grid_potential'

In [ ]:
rec_data[1].data.columns

## positions

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(12,4))
fig.suptitle('Reconstructed Positions')
axs[0].scatter(
    rec_data[1].data['RA'][::100],
    rec_data[1].data['DEC'][::100],
    s=3
)
axs[0].set_xlabel('RA')
axs[0].set_ylabel('DEC')

axs[1].scatter(
    rec_data[1].data['RA'][::100],
    rec_data[1].data['REDSHIFT'][::100],
    s=3
)
axs[1].set_xlabel('RA')
axs[1].set_ylabel('REDSHIFT')

axs[2].scatter(
    rec_data[1].data['DEC'][::100],
    rec_data[1].data['REDSHIFT'][::100],
    s=3
)
axs[1].set_xlabel('DEC')
axs[1].set_ylabel('REDSHIFT')
plt.tight_layout()

## displacement of tracers

In [ ]:
fig, axs = plt.subplots(1,4,figsize=(16,4))
fig.suptitle('tracers displacement')
comps = ['X', 'Y', 'Z']
for i, ax in enumerate(axs[:3]):
    c = comps[i]
    ax.hist(rec_data[1].data[f'S_{c}'], bins=50)
    ax.set_xlabel(f'$\psi_{c}$')

s_norm = np.linalg.norm(np.array([
    rec_data[1].data['S_X'],
    rec_data[1].data['S_Y'],
    rec_data[1].data['S_Z']
]), axis=0)

# Rimosso l'argomento non valido axs=0
axs[3].hist(s_norm, bins=50)
axs[3].set_xlabel('$||\psi||$')

## grid displacement and potential


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# 1. Load both datasets
with fits.open(saved_paths['grid_displacement']) as hdul:
    psi_data = hdul[0].data  # Shape: (3, Nx, Ny, Nz) or similar

with fits.open(saved_paths['grid_potential']) as pdul:
    phi_data = pdul[0].data  # Shape: (Nx, Ny, Nz)

# 2. Extract the slice at Z = slice
slice = 512
# Displacement components
step = 4
U = psi_data[::step, ::step, slice, 0]
V = psi_data[::step, ::step, slice, 1]

# Potential field at the same slice
# (Keep full resolution for the background image so it looks smooth)
phi_slice = phi_data[:, :, slice]

# 3. Plotting
fig, ax = plt.subplots(figsize=(4.5, 4.5))

# Plot the potential as a smooth background heatmap
# 'origin=lower' is crucial to keep FITS coordinate orientation consistent
im = ax.imshow(phi_slice, origin='lower', cmap='coolwarm', alpha=0.8)
fig.colorbar(im, ax=ax, label='Potential $\phi$', shrink=0.8, aspect=20,)

# Create matching X, Y coordinate mesh for the downsampled quiver arrows
# This ensures the arrows align perfectly with the background pixel grid
Nx, Ny = phi_slice.shape
X, Y = np.meshgrid(np.arange(0, Nx, step), np.arange(0, Ny, step), indexing='ij')

# Overplot the displacement arrows
ax.quiver(X, Y, U, V, color='black', scale=300.0, alpha=0.9)

ax.set_title("Displacement Field Overlaid on Potential ($\phi$), Slice Z=50")
ax.set_xlabel("X [Grid units]")
ax.set_ylabel("Y [Grid units]")

plt.show()

## pickle object

In [ ]:
import pickle
from zeldareco.BAOreconstruction.bao_reconstructor import BAOReconstructor

In [ ]:


with open(saved_paths['reconstructor_object'], 'rb') as f:
    reconstructor = pickle.load(f)

# Check what kind of object you just loaded
print(type(reconstructor))

In [ ]:
reconstructor.__dict__